# Dimensión 4 — Comparación de métricas de proyección de repartos y análisis de capacidad (versión corregida)

**Antes de ejecutar:** sube al panel de archivos de Colab (ícono de carpeta, a la izquierda) los cinco archivos: `delivery_events.csv`, `driver_monthly_metrics.csv`, `facilities.csv`, `trucks.csv` y `trailers.csv`. Luego ejecuta las celdas **en orden** (Entorno de ejecución → Ejecutar todas).

| Bloque | Qué hace | Sección del informe |
|---|---|---|
| 1 | Serie mensual, capacidad por recurso y Figuras 3.10–3.12 | Consideraciones de datos de entrada |
| 2 | Reducción de dimensiones (varianza, correlación, PCA) | Consideraciones de datos de entrada |
| 3 | Pruebas de estacionariedad (ADF, KPSS) y ACF/PACF | Características del modelo |
| 4 | **Modelo SARIMA(1,0,0)(1,0,0)[12]** (misma estructura del código entregado; cambios marcados con `CAMBIO`) | Bosquejo en código |
| 5 | Comparación de 6 configuraciones SARIMA | Características del modelo |
| 6 | Comparación con modelos de referencia (Seasonal Naïve, Holt-Winters, media) | Características del modelo |
| 7 | Diagnóstico de residuos (Ljung-Box) | Características del modelo |
| 8 | Proyección 2025 y análisis de capacidad | Bosquejo en código |

El código original entregado se conserva sin cambios en el archivo `DIMENSION 4.ipynb`.

In [ ]:
# =====================================================================
# BLOQUE 1: DATOS DE ENTRADA, CAPACIDAD OPERATIVA Y FIGURAS 3.10 - 3.12
# =====================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------------
# 1. CARGA DE TABLAS (archivos .csv subidos al panel lateral de Colab)
# ---------------------------------------------------------------------
delivery_events = pd.read_csv('delivery_events.csv')
driver_monthly_metrics = pd.read_csv('driver_monthly_metrics.csv')
facilities = pd.read_csv('facilities.csv')
trucks = pd.read_csv('trucks.csv')
trailers = pd.read_csv('trailers.csv')

# ---------------------------------------------------------------------
# 2. SERIE MENSUAL DE ENTREGAS PROGRAMADAS (2022 - 2024)
# ---------------------------------------------------------------------
delivery_events['scheduled_datetime'] = pd.to_datetime(delivery_events['scheduled_datetime'], errors='coerce')

# Solo eventos de entrega ('Delivery') y sin enero 2025 (mes incompleto)
deliv_only = delivery_events[delivery_events['event_type'] == 'Delivery'].copy()
deliv_36_meses = deliv_only[deliv_only['scheduled_datetime'] < '2025-01-01'].copy()

# Entregas por mes y promedio móvil de 3 meses
m_deliv = deliv_36_meses.groupby(deliv_36_meses['scheduled_datetime'].dt.to_period('M')).size()
m_deliv_df = m_deliv.to_frame(name='entregas')
m_deliv_df['mes_str'] = m_deliv_df.index.astype(str)
m_deliv_df['prom_movil_3m'] = m_deliv_df['entregas'].rolling(window=3).mean()

print(f"Meses analizados: {len(m_deliv_df)} | Total de entregas: {m_deliv_df['entregas'].sum():,}")
print(m_deliv_df['entregas'].describe().round(2))

# ---------------------------------------------------------------------
# 3. CAPACIDAD OPERATIVA CALCULADA CON LOS DATOS (cuello de botella)
# ---------------------------------------------------------------------
# Productividad mensual alta: percentil 95 de viajes completados por conductor
p95_viajes = driver_monthly_metrics['trips_completed'].quantile(0.95)

n_conductores = driver_monthly_metrics['driver_id'].nunique()
n_camiones = (trucks['status'] == 'Active').sum()
n_trailers = (trailers['status'] == 'Active').sum()
n_mantenimiento = (trucks['status'] == 'Maintenance').sum()

capacidades = pd.Series({
    'Conductores': n_conductores * p95_viajes,
    'Camiones activos': n_camiones * p95_viajes,
    'Tráileres activos': n_trailers * p95_viajes,
})
capacidad_ref = int(capacidades.min())   # el recurso más escaso limita toda la operación
muelles = facilities['dock_doors'].sum()

print(f"\nP95 de viajes por mes: {p95_viajes:.0f}")
print(f"Recursos: {n_conductores} conductores | {n_camiones} camiones activos "
      f"({n_mantenimiento} en mantenimiento) | {n_trailers} tráileres")
print(capacidades.astype(int).to_frame('Capacidad mensual (entregas)'))
print(f"Cuello de botella: {capacidades.idxmin()} -> capacidad de referencia = {capacidad_ref:,} entregas/mes")
print(f"Muelles: {muelles:,} en {len(facilities)} instalaciones | "
      f"entregas por muelle al mes: {(m_deliv_df['entregas'] / muelles).mean():.2f}")

# ---------------------------------------------------------------------
# 4. INDICADORES DE CAPACIDAD (Tabla 3.3)
# ---------------------------------------------------------------------
m_deliv_df['utilizacion'] = m_deliv_df['entregas'] / capacidad_ref * 100
m_deliv_df['brecha'] = capacidad_ref - m_deliv_df['entregas']   # (+) holgura | (-) demanda sobre la capacidad
meses_sobre = (m_deliv_df['entregas'] > capacidad_ref).sum()

print(f"\nUtilización media: {m_deliv_df['utilizacion'].mean():.2f}% "
      f"(mín. {m_deliv_df['utilizacion'].min():.2f}% | máx. {m_deliv_df['utilizacion'].max():.2f}%)")
print(f"Meses sobre la capacidad de referencia: {meses_sobre} de {len(m_deliv_df)} | "
      f"exceso máximo: {-m_deliv_df['brecha'].min():.0f} entregas")

# ---------------------------------------------------------------------
# 5. PANEL DE GRÁFICOS (Figuras 3.10, 3.11 y 3.12)
# ---------------------------------------------------------------------
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(3, 1, figsize=(14, 15))
x = np.arange(len(m_deliv_df))          # eje numérico: permite sombrear bien los meses sobre la capacidad
etiquetas = m_deliv_df['mes_str']

# --- Figura 3.10: Evolución mensual y promedio móvil ---
axes[0].plot(x, m_deliv_df['entregas'], marker='o', color='#1f77b4', linewidth=2, label='Entregas Mensuales')
axes[0].plot(x, m_deliv_df['prom_movil_3m'], color='#ff7f0e', linestyle='--', linewidth=2.5, label='Promedio Móvil (3 Meses)')
axes[0].set_title('Evolución Mensual de Entregas Programadas y Promedio Móvil (2022-2024)', fontsize=13, fontweight='bold', pad=12)
axes[0].set_ylabel('Número de Entregas', fontsize=11)
axes[0].set_xticks(x)
axes[0].set_xticklabels(etiquetas, rotation=75)
axes[0].set_ylim(2100, 2600)
axes[0].legend(loc='upper center', ncol=2)

# --- Figura 3.11: Demanda vs. capacidad operativa de referencia ---
axes[1].plot(x, m_deliv_df['entregas'], marker='s', color='#2ca02c', linewidth=2, label='Demanda Mensual (Entregas)')
axes[1].axhline(y=capacidad_ref, color='#d62728', linestyle='-', linewidth=2.5,
                label=f'Capacidad de Referencia ({capacidad_ref:,} entregas)')
axes[1].fill_between(x, m_deliv_df['entregas'], capacidad_ref,
                     where=(m_deliv_df['entregas'] > capacidad_ref), interpolate=True,
                     color='#d62728', alpha=0.3, label='Demanda sobre la capacidad')
axes[1].set_title('Demanda Mensual vs. Capacidad Operativa de Referencia', fontsize=13, fontweight='bold', pad=12)
axes[1].set_ylabel('Número de Entregas', fontsize=11)
axes[1].set_xticks(x)
axes[1].set_xticklabels(etiquetas, rotation=75)
axes[1].set_ylim(2100, 2600)
axes[1].legend(loc='upper center', ncol=3)

# --- Figura 3.12: Distribución de la tasa de utilización ---
sns.histplot(m_deliv_df['utilizacion'], kde=True, ax=axes[2], color='#9467bd', bins=12)
axes[2].axvline(x=100, color='#d62728', linestyle='--', linewidth=2, label='100% Capacidad Límite')
axes[2].set_title('Distribución de la Tasa de Utilización de Capacidad Operativa (%)', fontsize=13, fontweight='bold', pad=12)
axes[2].set_xlabel('Tasa de Utilización (%)', fontsize=11)
axes[2].set_ylabel('Frecuencia (Meses)', fontsize=11)
axes[2].legend(loc='upper left')

plt.tight_layout()
plt.savefig('panel_dimension4.png', dpi=300)
plt.show()

### Bloque 2 — Reducción de dimensiones
Antes de modelar se revisa qué variables aportan información distinta. Se arman 20 variables mensuales candidatas con las tablas de origen y se aplican tres filtros: (1) varianza cero, (2) redundancia por correlación y (3) análisis de componentes principales (PCA).

In [ ]:
# =====================================================================
# BLOQUE 2: REDUCCIÓN DE DIMENSIONES (selección de variables de entrada)
# =====================================================================
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ---------------------------------------------------------------------
# 1. VARIABLES MENSUALES CANDIDATAS (36 meses x 20 variables)
# ---------------------------------------------------------------------
d = deliv_36_meses.copy()
d['actual_datetime'] = pd.to_datetime(d['actual_datetime'], errors='coerce')
d['mes'] = d['scheduled_datetime'].dt.to_period('M')
d['retraso_min'] = (d['actual_datetime'] - d['scheduled_datetime']).dt.total_seconds() / 60

var_entregas = d.groupby('mes').agg(
    entregas=('event_id', 'size'),
    detencion_prom=('detention_minutes', 'mean'),
    puntualidad=('on_time_flag', 'mean'),
    retraso_prom=('retraso_min', 'mean'),
    instalaciones_destino=('facility_id', 'nunique'))

dm = driver_monthly_metrics.copy()
dm['mes'] = pd.to_datetime(dm['month']).dt.to_period('M')
var_conductores = dm.groupby('mes').agg(
    conductores=('driver_id', 'nunique'),
    viajes_conductor=('trips_completed', 'mean'),
    viajes_totales=('trips_completed', 'sum'),
    millas=('total_miles', 'sum'),
    ingresos=('total_revenue', 'sum'),
    mpg=('average_mpg', 'mean'),
    galones=('total_fuel_gallons', 'sum'),
    otd_conductor=('on_time_delivery_rate', 'mean'),
    ralenti=('average_idle_hours', 'mean'))

X = var_entregas.join(var_conductores)
X['dias_mes'] = X.index.days_in_month
X['utilizacion'] = m_deliv_df['utilizacion']
X['brecha'] = m_deliv_df['brecha']
X['entregas_muelle'] = X['entregas'] / muelles
X['muelles_totales'] = muelles
X['n_instalaciones'] = len(facilities)
print(f"Variables candidatas: {X.shape[1]} | Meses: {X.shape[0]}")

# ---------------------------------------------------------------------
# 2. FILTRO 1: VARIANZA CERO (variables que no cambian en el tiempo)
# ---------------------------------------------------------------------
desv = X.std()
var_cero = desv[desv < 1e-9].index.tolist()
print("\nFiltro 1 - Varianza cero (se eliminan):", var_cero)
X1 = X.drop(columns=var_cero)

# ---------------------------------------------------------------------
# 3. FILTRO 2: REDUNDANCIA (matriz de correlación)
# ---------------------------------------------------------------------
corr = X1.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, square=True,
            annot_kws={'size': 8}, cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlación de las Variables Mensuales Candidatas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('matriz_correlacion_dim4.png', dpi=300)
plt.show()

print("Filtro 2 - Correlación de cada variable con las entregas mensuales:")
print(corr['entregas'].drop('entregas').sort_values(key=abs, ascending=False).round(3))

# utilización, brecha y entregas por muelle son la misma variable reescalada (|r| = 1)
redundantes = ['utilizacion', 'brecha', 'entregas_muelle']
X2 = X1.drop(columns=redundantes)
print("\nSe eliminan por ser transformaciones lineales de 'entregas':", redundantes)

# ---------------------------------------------------------------------
# 4. FILTRO 3: ANÁLISIS DE COMPONENTES PRINCIPALES (PCA)
# ---------------------------------------------------------------------
Z = StandardScaler().fit_transform(X2)          # se estandariza: las variables tienen escalas distintas
pca = PCA().fit(Z)

var_exp = pd.DataFrame({
    'Varianza explicada (%)': pca.explained_variance_ratio_ * 100,
    'Acumulada (%)': np.cumsum(pca.explained_variance_ratio_) * 100,
    'Autovalor': pca.explained_variance_},
    index=[f'PC{i + 1}' for i in range(len(pca.explained_variance_ratio_))]).round(2)
print(f"\nFiltro 3 - PCA sobre {X2.shape[1]} variables:")
print(var_exp.head(6))
print(f"Componentes con autovalor > 1: {(pca.explained_variance_ > 1).sum()}")

cargas = pd.DataFrame(pca.components_[:2].T, index=X2.columns, columns=['PC1', 'PC2']).round(2)
print("\nCargas de los dos primeros componentes:")
print(cargas.sort_values('PC1', key=abs, ascending=False))

print("\nConclusión: PC1 agrupa el volumen operativo (entregas, viajes, millas, ingresos, galones, días del mes).")
print("La serie de entregas mensuales resume ese factor, por eso el modelo SARIMA usa una sola serie.")

### Bloque 3 — Estacionariedad y orden del modelo
ADF (hipótesis nula: la serie tiene raíz unitaria) y KPSS (hipótesis nula: la serie es estacionaria) justifican d = 0. La ACF y la PACF justifican los órdenes p = 1 y P = 1.

In [ ]:
# =====================================================================
# BLOQUE 3: PRUEBAS DE ESTACIONARIEDAD Y FUNCIONES DE AUTOCORRELACIÓN
# =====================================================================
import warnings
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

serie = m_deliv_df['entregas'].astype(float)

# ADF: H0 = la serie tiene raíz unitaria (no estacionaria). p < 0.05 -> estacionaria
adf_stat, adf_p, adf_lags, adf_nobs, adf_crit, _ = adfuller(serie, regression='c', autolag='AIC')

# KPSS: H0 = la serie es estacionaria. p > 0.05 -> estacionaria
with warnings.catch_warnings():
    warnings.simplefilter('ignore')     # KPSS avisa cuando el p-valor sale de su tabla (p >= 0.10)
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(serie, regression='c', nlags='auto')

print(f"ADF : estadístico = {adf_stat:.3f} | p-valor = {adf_p:.6f} | valor crítico 5% = {adf_crit['5%']:.3f}")
print(f"KPSS: estadístico = {kpss_stat:.3f} | p-valor = {kpss_p:.3f} | valor crítico 5% = {kpss_crit['5%']:.3f}")
if adf_p < 0.05 and kpss_p > 0.05:
    print("Conclusión: ambas pruebas indican que la serie es estacionaria -> d = 0 y se incluye constante")
else:
    print("Conclusión: las pruebas no coinciden; revisar diferenciación")

# ACF y PACF (bandas = intervalo de confianza 95%)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
plot_acf(serie, lags=15, ax=ax[0])
plot_pacf(serie, lags=15, ax=ax[1], method='ywm')
ax[0].set_title('Función de Autocorrelación (ACF)', fontweight='bold')
ax[1].set_title('Función de Autocorrelación Parcial (PACF)', fontweight='bold')
for a in ax:
    a.set_xlabel('Rezago (meses)')
plt.tight_layout()
plt.savefig('acf_pacf_dim4.png', dpi=300)
plt.show()

### Bloque 4 — Modelo SARIMA(1,0,0)(1,0,0)[12] (código corregido)
Es el mismo código entregado, con las mismas secciones 1 a 6 y los mismos nombres de variables. Cada línea modificada lleva el comentario `CAMBIO`.

In [ ]:
# =====================================================================
# BLOQUE 4: MODELO SARIMA(1,0,0)(1,0,0)[12] - VERSIÓN CORREGIDA
# Misma estructura del código entregado. Las líneas modificadas dicen "CAMBIO".
# =====================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Cargar datos y construir la serie mensual directamente de la tabla
df_events = pd.read_csv('delivery_events.csv')
df_events['scheduled_datetime'] = pd.to_datetime(df_events['scheduled_datetime'])  # CAMBIO: fecha programada (igual que el Bloque 1)
df_events = df_events[df_events['event_type'] == 'Delivery']                      # CAMBIO: solo entregas (antes sumaba también los recojos)
df_events = df_events[df_events['scheduled_datetime'] < '2025-01-01']              # CAMBIO: se excluye enero 2025 (mes incompleto)

# Crear la serie temporal mensual
monthly_deliveries = df_events.resample('MS', on='scheduled_datetime').size().to_frame(name='delivery_count')

# Asegurar que el índice sea DatetimeIndex
if not isinstance(monthly_deliveries.index, pd.DatetimeIndex):
    monthly_deliveries.index = pd.to_datetime(monthly_deliveries.index)

print(f"Serie mensual: {len(monthly_deliveries)} meses | {monthly_deliveries['delivery_count'].sum():,} entregas")

# 2. Definir el rango de entrenamiento (2022-2023) y prueba (2024)
train_data = monthly_deliveries.loc['2022-01-01':'2023-12-01', 'delivery_count']
test_data = monthly_deliveries.loc['2024-01-01':'2024-12-01', 'delivery_count']

# 3. Entrenar el modelo SARIMA(1,0,0)(1,0,0)[12]
model = SARIMAX(train_data,
                order=(1,0,0),
                seasonal_order=(1,0,0,12),
                trend='c')                     # CAMBIO: constante (nivel medio). Se quitaron enforce_stationarity/invertibility=False
model_fit = model.fit(disp=False, maxiter=200)  # CAMBIO: más iteraciones para asegurar la convergencia
print(model_fit.summary())

# CAMBIO: nivel medio que aprendió el modelo = c / ((1 - phi) * (1 - Phi))
p = model_fit.params
print(f"Media implícita del modelo: {p['intercept'] / ((1 - p['ar.L1']) * (1 - p['ar.S.L12'])):.1f} entregas/mes")

# 4. Pronosticar las entregas para 2024
pronostico = model_fit.get_forecast(steps=len(test_data))   # CAMBIO: get_forecast entrega también el intervalo de confianza
forecast_2024 = pronostico.predicted_mean
ic_2024 = pronostico.conf_int(alpha=0.05)

# 5. Graficar los resultados
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data, label='Entregas Reales (2022-2023)', color='gray')
plt.plot(test_data.index, test_data, label='Entregas Reales (2024)', color='blue')
plt.plot(forecast_2024.index, forecast_2024, label='Entregas Pronosticadas (2024)', color='red', linestyle='--')
plt.fill_between(ic_2024.index, ic_2024.iloc[:, 0], ic_2024.iloc[:, 1],
                 color='red', alpha=0.12, label='Intervalo de confianza 95%')   # CAMBIO
plt.title('Pronóstico de Entregas Mensuales con Modelo SARIMA')
plt.xlabel('Fecha')
plt.ylabel('Cantidad de Entregas')
plt.legend(loc='lower left')
plt.grid(True)
plt.tight_layout()
plt.savefig('sarima_2024_dim4.png', dpi=300)
plt.show()

# 6. Calcular métricas de evaluación
actuals = test_data
predictions = forecast_2024

mae = mean_absolute_error(actuals, predictions)
rmse = np.sqrt(mean_squared_error(actuals, predictions))

def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero_actuals_mask = y_true != 0
    if non_zero_actuals_mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[non_zero_actuals_mask] - y_pred[non_zero_actuals_mask]) / y_true[non_zero_actuals_mask])) * 100

mape = mean_absolute_percentage_error(actuals, predictions)

print(f"\nMétricas de Evaluación del Pronóstico para 2024:")
print(f"MAE (Mean Absolute Error): {mae:.2f}")
print(f"RMSE (Root Mean Squared Error): {rmse:.2f}")
print(f"MAPE (Mean Absolute Percentage Error): {mape:.2f}%")

### Bloque 5 — Comparación de 6 configuraciones SARIMA
Todas se entrenan con 2022–2023 y se evalúan en 2024. El AIC se calcula con los datos de entrenamiento (menor = mejor ajuste penalizando la complejidad); MAE, RMSE y MAPE miden el error de pronóstico en 2024.

In [ ]:
# =====================================================================
# BLOQUE 5: COMPARACIÓN DE 6 CONFIGURACIONES SARIMA
# =====================================================================
import warnings

def formato(orden):
    return '(' + ','.join(str(v) for v in orden) + ')'

configuraciones = [
    ((1, 0, 0), (1, 0, 0, 12), 'n'),   # versión entregada: sin constante
    ((1, 0, 0), (1, 0, 0, 12), 'c'),   # modelo elegido
    ((1, 0, 0), (0, 0, 0, 0),  'c'),   # solo término autorregresivo simple
    ((0, 0, 0), (1, 0, 0, 12), 'c'),   # solo término estacional
    ((2, 0, 0), (1, 0, 0, 12), 'c'),   # AR de orden 2 + estacional
    ((2, 0, 0), (0, 0, 0, 0),  'c'),   # AR de orden 2 sin estacional
]

filas = []
for orden, orden_est, tendencia in configuraciones:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        ajuste = SARIMAX(train_data, order=orden, seasonal_order=orden_est, trend=tendencia).fit(disp=False, maxiter=200)
    pred = ajuste.get_forecast(steps=len(test_data)).predicted_mean
    nombre = f"SARIMA{formato(orden)}{formato(orden_est[:3])}"
    if orden_est[3]:
        nombre += f"[{orden_est[3]}]"
    nombre += ' con constante' if tendencia == 'c' else ' sin constante'
    filas.append({'Configuración': nombre,
                  'AIC': ajuste.aic,
                  'MAE': mean_absolute_error(test_data, pred),
                  'RMSE': np.sqrt(mean_squared_error(test_data, pred)),
                  'MAPE (%)': mean_absolute_percentage_error(test_data, pred)})

tabla_config = pd.DataFrame(filas).round(2)
print(tabla_config.to_string(index=False))
print(f"\nMenor AIC : {tabla_config.loc[tabla_config['AIC'].idxmin(), 'Configuración']}")
print(f"Menor RMSE: {tabla_config.loc[tabla_config['RMSE'].idxmin(), 'Configuración']}")

### Bloque 6 — Comparación de métricas con modelos de referencia
Un modelo de pronóstico solo aporta valor si supera a alternativas simples evaluadas sobre la misma serie y el mismo periodo de prueba (2024).

In [ ]:
# =====================================================================
# BLOQUE 6: COMPARACIÓN DE MÉTRICAS CON MODELOS DE REFERENCIA (2024)
# =====================================================================
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Seasonal Naïve: cada mes de 2024 se pronostica con el mismo mes de 2023
pred_snaive = pd.Series(train_data.iloc[-12:].values, index=test_data.index)

# Holt-Winters aditivo: nivel + tendencia + estacionalidad (suavizamiento exponencial triple)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    hw_fit = ExponentialSmoothing(train_data, trend='add', seasonal='add', seasonal_periods=12,
                                  initialization_method='estimated').fit()
pred_hw = pd.Series(hw_fit.forecast(len(test_data)).values, index=test_data.index)

# Media histórica: promedio de 2022-2023 para todos los meses
pred_media = pd.Series(train_data.mean(), index=test_data.index)

modelos = {
    'SARIMA(1,0,0)(1,0,0)[12]': forecast_2024,
    'Seasonal Naïve': pred_snaive,
    'Holt-Winters aditivo': pred_hw,
    'Media histórica': pred_media,
}
tabla_comp = pd.DataFrame({
    nombre: {'MAE': mean_absolute_error(test_data, pred),
             'RMSE': np.sqrt(mean_squared_error(test_data, pred)),
             'MAPE (%)': mean_absolute_percentage_error(test_data, pred)}
    for nombre, pred in modelos.items()}).T.round(2).sort_values('RMSE')
print(tabla_comp)

# Gráfico comparativo
plt.figure(figsize=(14, 6))
plt.plot(test_data.index, test_data, color='black', linewidth=2.5, marker='o', label='Entregas reales 2024')
estilos = {'SARIMA(1,0,0)(1,0,0)[12]': ('red', '--'), 'Seasonal Naïve': ('#1f77b4', ':'),
           'Holt-Winters aditivo': ('#2ca02c', '-.'), 'Media histórica': ('gray', '-')}
for nombre, pred in modelos.items():
    color, linea = estilos[nombre]
    plt.plot(test_data.index, pred, color=color, linestyle=linea, linewidth=2, label=nombre)
plt.title('Comparación de Pronósticos 2024: SARIMA vs. Modelos de Referencia', fontweight='bold')
plt.ylabel('Cantidad de Entregas')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.savefig('comparacion_modelos_dim4.png', dpi=300)
plt.show()

### Bloque 7 — Diagnóstico de residuos
Si el modelo capturó bien la dinámica de la serie, sus residuos no deben tener autocorrelación (prueba de Ljung-Box: p > 0.05).

In [ ]:
# =====================================================================
# BLOQUE 7: DIAGNÓSTICO DE RESIDUOS DEL SARIMA
# =====================================================================
from statsmodels.stats.diagnostic import acorr_ljungbox

model_fit.plot_diagnostics(figsize=(14, 9))
plt.tight_layout()
plt.show()

# Ljung-Box: H0 = los residuos no están autocorrelacionados (ruido blanco). p > 0.05 -> modelo adecuado
ljung_box = acorr_ljungbox(model_fit.resid, lags=[6, 12])
print(ljung_box.round(3))

# Cobertura del intervalo de confianza en 2024
dentro = ((test_data >= ic_2024.iloc[:, 0]) & (test_data <= ic_2024.iloc[:, 1])).sum()
print(f"\nMeses de 2024 dentro del intervalo de confianza 95%: {dentro} de {len(test_data)}")

### Bloque 8 — Proyección 2025 y análisis de capacidad
Validado el modelo, se reentrena con los 36 meses (2022–2024) y se proyecta 2025. La proyección se contrasta con la capacidad de referencia calculada en el Bloque 1.

In [ ]:
# =====================================================================
# BLOQUE 8: PROYECCIÓN 2025 Y ANÁLISIS DE CAPACIDAD DE DISTRIBUCIÓN
# =====================================================================
# 1. Reentrenar el mismo modelo con los 36 meses completos
full_model = SARIMAX(monthly_deliveries['delivery_count'],
                     order=(1,0,0),
                     seasonal_order=(1,0,0,12),
                     trend='c')
full_fit = full_model.fit(disp=False, maxiter=200)

# 2. Proyectar enero-diciembre 2025 con intervalo de confianza 95%
proyeccion = full_fit.get_forecast(steps=12)
proy_2025 = proyeccion.predicted_mean
ic_2025 = proyeccion.conf_int(alpha=0.05)

# 3. Indicadores de capacidad proyectados
tabla_2025 = pd.DataFrame({
    'Pronóstico': proy_2025.round(0),
    'Lím. inf. 95%': ic_2025.iloc[:, 0].round(0),
    'Lím. sup. 95%': ic_2025.iloc[:, 1].round(0),
    'Utilización (%)': (proy_2025 / capacidad_ref * 100).round(2),
    'Brecha': (capacidad_ref - proy_2025).round(0),   # (+) holgura | (-) déficit de capacidad
})
tabla_2025.index = tabla_2025.index.strftime('%Y-%m')
print(tabla_2025)

meses_deficit = (proy_2025 > capacidad_ref).sum()
meses_riesgo = (ic_2025.iloc[:, 1] > capacidad_ref).sum()
deficit_max = (proy_2025 - capacidad_ref).max()
camiones_puntual = int(np.ceil(max(deficit_max, 0) / p95_viajes))
camiones_ic = int(np.ceil(max(ic_2025.iloc[:, 1].max() - capacidad_ref, 0) / p95_viajes))

print(f"\nDemanda proyectada 2025: {proy_2025.sum():,.0f} entregas (media {proy_2025.mean():,.1f} por mes)")
print(f"Utilización media proyectada: {(proy_2025 / capacidad_ref * 100).mean():.2f}%")
print(f"Meses con pronóstico sobre la capacidad: {meses_deficit} de 12 | déficit máximo: {deficit_max:.0f} entregas")
print(f"Meses en que el límite superior del IC 95% supera la capacidad: {meses_riesgo} de 12")
print(f"Camiones adicionales necesarios: {camiones_puntual} (pronóstico) | {camiones_ic} (límite superior IC 95%) "
      f"| camiones en mantenimiento disponibles: {n_mantenimiento}")

# 4. Gráfico: histórico, proyección 2025 y capacidad
plt.figure(figsize=(14, 7))
plt.plot(monthly_deliveries.index, monthly_deliveries['delivery_count'], color='gray', marker='o', markersize=4,
         label='Entregas reales (2022-2024)')
plt.plot(proy_2025.index, proy_2025, color='red', linestyle='--', marker='o', markersize=4,
         label='Proyección SARIMA 2025')
plt.fill_between(ic_2025.index, ic_2025.iloc[:, 0], ic_2025.iloc[:, 1], color='red', alpha=0.12,
                 label='Intervalo de confianza 95%')
plt.axhline(y=capacidad_ref, color='#d62728', linewidth=2.5, label=f'Capacidad de referencia ({capacidad_ref:,})')
plt.title('Proyección de Entregas 2025 vs. Capacidad Operativa de Referencia', fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Cantidad de Entregas')
plt.legend(loc='lower left')
plt.grid(True)
plt.tight_layout()
plt.savefig('proyeccion_2025_capacidad_dim4.png', dpi=300)
plt.show()